<h1>IMPORTAÇÃO DE BIBLIOTECAS</h1>

In [ ]:
import pandas as pd
import numpy as np
from scipy.stats import skew
from scipy.stats import kurtosis
from scipy.stats import mode

from scipy.fft import fft, fftfreq


<h1>Extração de Features - MACHINE LEARNING</h1>

In [ ]:
preprocessing_path = "./preprocessing/"

segmented_dataset_fileName = "03-segmented_dataset.csv"
features_dataset_filename = "04-features_dataset.csv"


def extract_time_features(data):
    time_features = {
        "amplitude": np.max(data) - np.min(data),
        "energy": np.sum(data**2),
        "kurtosis": kurtosis(data, fisher=False),
        "max": np.max(data),
        "mean": np.mean(data),
        "median": np.median(data),
        "min": np.min(data),
        "mode": mode(data, keepdims=False).mode,
        "rms": np.sqrt(np.mean(data**2)),
        "skewness": skew(data),
        "std": np.std(data),
        "variance": np.var(data)
    }
    return time_features

def load_rawData(dataFile):
    df_rawData = pd.read_csv(dataFile, sep=",",decimal=".") # lendo arquivo

    df_rawData["module"] = np.sqrt(np.pow(df_rawData["Q_data"],2) + np.pow(df_rawData["I_data"],2)) # Calculando modulo do sinal
    df_rawData["fase"] = np.atan(df_rawData["Q_data"]/df_rawData["I_data"]) # Calculando fase do sinal

    df_rawData = df_rawData[df_rawData["module"] > 0] # Filtrando valores não nulos

    return df_rawData

def extract_freq_features(data):
    df_rawData = data.copy()
    df_rawData["f(t)"] = data["I_data"] + data["Q_data"]*1j

    fs = 10000 #amostras por segundo 10kHz
    N = df_rawData.shape[0]

    Z = fft(df_rawData["f(t)"])
    freq = fftfreq(N, 1/fs)

    # Apenas frequências positivas
    mask = freq >= 0

    freq = freq[mask]
    magnitude = np.abs(Z)[mask]


    # Frequência dominante
    idx_dominante = np.argmax(magnitude)
    frequencia_dominante = freq[np.argmax(magnitude)]
    magnitude_freq_dominante = magnitude[idx_dominante]

    # Frequência média ponderada pela magnitude
    frequencia_media = np.average(freq, weights=magnitude) 

    # Frequência mediana ponderada
    magnitude_acumulada = np.cumsum(magnitude)
    limite = magnitude_acumulada[-1] / 2

    idx_mediana = np.searchsorted(magnitude_acumulada, limite)
    frequencia_mediana = freq[idx_mediana]

    # Energia espectral
    energia_espectral = np.sum(magnitude ** 2)

    # Largura de banda
    # RMS bandwidth
    frequencia_centroide = frequencia_media
    largura_banda = np.sqrt(
        np.sum(
            magnitude * (freq - frequencia_centroide) ** 2
        ) / np.sum(magnitude)
    )

    # Curtose e assimetria do espectro
    assimetria = skew(magnitude)
    curtose = kurtosis(
        magnitude,
        fisher=False
    )

    return {
        "bandwidth": largura_banda,
        "dominant_frequency": frequencia_dominante,
        "dominant_frequency_magnitude": magnitude_freq_dominante,
        "energy_spectral": energia_espectral,
        "frequency_mean": frequencia_media,
        "frequency_median": frequencia_mediana,
        "kurtosis": curtose,
        "skewness": assimetria
    }


def extract_features_from_dataset(dataDetails):
    dataDetails = dataDetails.copy()
    total = dataDetails.shape[0]
    for indice, data in dataDetails.iterrows():
        data_rawData = load_rawData(dataFile=data["rawData"])

        # features domínio do tempo:
        I_features = extract_time_features(data_rawData["I_data"])
        for feature, value in I_features.items(): dataDetails.loc[indice, "I_" + feature] = value

        Q_features = extract_time_features(data_rawData["Q_data"])
        for feature, value in Q_features.items(): dataDetails.loc[indice, "Q_" + feature] = value

        module_features = extract_time_features(data_rawData["module"])
        for feature, value in module_features.items(): dataDetails.loc[indice, "module_" + feature] = value

        fase_features = extract_time_features(data_rawData["fase"])
        for feature, value in fase_features.items(): dataDetails.loc[indice, "fase_" + feature] = value

        # features domínio da freqência:
        data_fft_features = extract_freq_features(data_rawData)
        for feature, value in data_fft_features.items(): dataDetails.loc[indice, "fft_" + feature] = value

        print(f"\r{((indice+1) / total) * 100:.2f}% concluído!", end="")
    return dataDetails

segmented_dataset = pd.read_csv(preprocessing_path+segmented_dataset_fileName)

print("Iniciando processo de extração....")
df_dataFeatures = extract_features_from_dataset(segmented_dataset)
df_dataFeatures.to_csv(preprocessing_path + features_dataset_filename, index=False)


Iniciando processo de extração....
99.99% concluído!

In [3]:
df_dataFeatures.describe()

,motorSpeed,load,I_amplitude,I_energy,I_kurtosis,I_max,I_mean,I_median,I_min,I_mode,...,fase_std,fase_variance,fft_bandwidth,fft_dominant_frequency,fft_dominant_frequency_magnitude,fft_energy_spectral,fft_frequency_mean,fft_frequency_median,fft_kurtosis,fft_skewness
count,13830.000000,13830.000000,13830.000000,13830.000000,13830.000000,13830.000000,13830.000000,13830.000000,13830.000000,13830.000000,...,13830.000000,13830.000000,13830.000000,13830.0,13830.000000,1.383000e+04,13830.000000,13830.000000,13830.000000,13830.000000
mean,1000.650759,1.247289,4.600611,74500.364166,2.756399,4.856370,2.588166,2.562837,0.255759,2.568327,...,0.226961,0.053060,818.294275,0.0,36537.703036,1.394427e+09,464.191280,163.530098,4614.804895,66.702571
std,316.272827,0.852880,0.617040,4193.459151,0.362579,0.383192,0.099479,0.122037,0.325675,1.178642,...,0.039351,0.019788,66.281220,0.0,1407.054936,6.736334e+07,71.820979,48.155937,106.483383,1.093183
min,500.000000,0.000000,0.010212,2322.016955,1.773020,0.488293,0.481844,0.483187,-0.006993,-0.001887,...,0.002899,0.000008,606.211482,0.0,6770.620673,4.584181e+07,247.490687,0.000000,3211.640088,53.025879
25%,700.000000,0.500000,4.355454,72809.132942,2.484575,4.613975,2.586335,2.510286,-0.001887,2.188605,...,0.203941,0.041592,774.562968,0.0,36516.482089,1.382270e+09,416.536086,127.987201,4564.007000,66.169986
50%,1000.000000,1.000000,4.779255,74579.764132,2.625663,4.935655,2.593242,2.561346,0.141082,2.525604,...,0.219076,0.047994,816.254213,0.0,36611.651499,1.395631e+09,465.968256,164.983502,4617.197488,66.722983
75%,1300.000000,2.000000,5.065193,76344.748267,2.956872,5.185851,2.599691,2.637937,0.386172,2.842178,...,0.247252,0.061133,859.034426,0.0,36693.098932,1.409702e+09,511.812958,199.980002,4707.122289,67.653116
max,1500.000000,2.500000,5.197950,99593.855272,3.842892,5.196064,2.726682,2.862602,1.249093,5.185851,...,0.792183,0.627554,1203.729567,0.0,41321.419357,1.869569e+09,738.272902,346.000000,4998.975496,70.696274


<h1>Criação das imagens - MACHINE LEARNING CNN</h1>

In [25]:
import matplotlib
matplotlib.use('agg')
import matplotlib.pyplot as plt
import os
import shutil

import scipy.signal


In [ ]:
path_sinalFigures = preprocessing_path + "images/" + "sinal/"
path_fourierFigures = preprocessing_path + "images/" + "fourier/"
path_spectogramFigures = preprocessing_path + "images/" + "spectogram/"

images_dataset_filename = "04.1-images_dataset.csv"

def limparPasta(file):
    for arquivo in os.listdir(file):
        caminho = os.path.join(file, arquivo)
        if os.path.isfile(caminho) or os.path.islink(caminho):
            os.remove(caminho)
        elif os.path.isdir(caminho):
            shutil.rmtree(caminho)

limparPasta(path_sinalFigures)
limparPasta(path_fourierFigures)
limparPasta(path_spectogramFigures)

def global_features(features_dataset):
    ymin = np.min(features_dataset['module_min'])
    ymax = np.max(features_dataset['module_max'])
    fase_min = np.min(features_dataset['fase_min'])
    fase_max = np.max(features_dataset['fase_max'])
    fft_max_frequency_magnitude = np.max(features_dataset['fft_dominant_frequency_magnitude'])
    return ymin, ymax, fase_min, fase_max, fft_max_frequency_magnitude

ymin_global, ymax_global, \
fase_min_global, fase_max_global, \
fft_max_frequency_magnitude \
= global_features(df_dataFeatures)

def create_sinalPlot(rawData, fileName):
    fig, ax = plt.subplots(
        figsize=(4, 3),
        dpi=100
    )
    
    x = np.arange(rawData.shape[0])

    ax.plot(x, rawData["module"], "g")

    ax2 = ax.twinx()
    ax2.plot(x, rawData["fase"])

    ax.set_ylim(ymin_global, ymax_global)
    ax2.set_ylim(fase_min_global, fase_max_global)

    ax.set_xlim(0, len(x))
    ax2.set_xlim(0, len(x))

    # Remove os valores dos eixos
    ax.set_xticks([])
    ax.set_yticks([])

    # Remove bordas
    for spine in ax.spines.values():
        spine.set_visible(False)

    # Ocupa toda a imagem
    fig.subplots_adjust(
        left=0,
        right=1,
        bottom=0,
        top=1
    )

    # Salva em 800 x 600 pixels
    fig.savefig(
        fileName,
        dpi=100,
        bbox_inches=None,
        pad_inches=0
    )

    plt.close(fig)

def create_fourierPlot(rawData, fileName):
    # Frequência de amostragem
    fs = 10000  # Hz

    sinal = rawData["I_data"] + rawData["Q_data"]*1j

    # Número de amostras
    N = len(rawData)

    # FFT
    fft = np.fft.fft(sinal)

    # Eixo de frequências
    frequencias = np.fft.fftfreq(N, 1/fs)

    # Magnitude
    magnitude = np.abs(fft)

    # Pega somente frequências positivas
    mask = np.abs(frequencias) >= 0

    fig, ax = plt.subplots(
        figsize=(4, 3),
        dpi=100
    )

    ax.plot(frequencias[mask], magnitude[mask])
    ax.set_ylim(0, fft_max_frequency_magnitude)
    ax.set_xlim(-fs/2, fs/2)
    ax.set_xticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    fig.subplots_adjust(
        left=0,
        right=1,
        bottom=0,
        top=1
    )
    fig.savefig(
        fileName,
        dpi=100,
        bbox_inches=None,
        pad_inches=0
    )
    plt.close(fig)

def create_spectogramPlot(rawData, fileName):
    fs = 10000  # frequência de amostragem

    # Sinal complexo
    sinal = rawData["I_data"].to_numpy() + \
            1j * rawData["Q_data"].to_numpy()

    # Parâmetros do espectrograma
    nperseg = 256
    noverlap = 128

    # Calcula o espectrograma
    f, t, Sxx = scipy.signal.stft(
        sinal,
        fs=fs,
        nperseg=nperseg,
        noverlap=noverlap,
        return_onesided=False
    )

    # Magnitude
    magnitude = np.abs(Sxx)

    fig, ax = plt.subplots(
        figsize=(4, 3),
        dpi=100
    )

    ax.pcolormesh(
        t,
        f,
        magnitude,
        shading="gouraud"
    )

    # Limites
    ax.set_ylim(-fs/2, fs / 2)

    # Remove eixos
    ax.set_xticks([])
    ax.set_yticks([])

    # Remove bordas
    for spine in ax.spines.values():
        spine.set_visible(False)

    fig.subplots_adjust(
        left=0,
        right=1,
        bottom=0,
        top=1
    )

    fig.savefig(
        fileName,
        dpi=100,
        bbox_inches=None,
        pad_inches=0
    )

    plt.close(fig)

def generate_figures_from_dataset(dataDetails):
    dataDetails = dataDetails.copy()
    total = dataDetails.shape[0]
    for indice, data in dataDetails.iterrows():
        data_rawData = load_rawData(dataFile=data["rawData"])

        fileName = dataDetails.loc[indice, 'dataName']
        dataDetails.loc[indice, "figure_sinal_path"] = path_sinalFigures + fileName + ".png"
        create_sinalPlot(data_rawData, dataDetails.loc[indice, "figure_sinal_path"])

        dataDetails.loc[indice, "figure_fourier_path"] = path_fourierFigures + fileName + ".png"
        create_fourierPlot(data_rawData, dataDetails.loc[indice, "figure_fourier_path"])

        dataDetails.loc[indice, "figure_spectogram_path"] = path_spectogramFigures + fileName + ".png"
        create_spectogramPlot(data_rawData, dataDetails.loc[indice, "figure_spectogram_path"])

        print(f"\r{((indice+1) / total) * 100:.2f}% concluído!", end="")
    return dataDetails

df_dataFeaturesWithImages = generate_figures_from_dataset(segmented_dataset)
df_dataFeaturesWithImages.to_csv(preprocessing_path + images_dataset_filename)

100.00% concluído!